# Phase 5 — Evaluation & Error Analysis

**Course**: CS5143 Natural Language Processing — Spring 2026, FAST-NUCES  
**Student**: Muhammad Azhar (24K-7606)

This notebook evaluates the trained XLM-R model on English and Spanish test sets,
produces per-entity F1 scores using seqeval, compares against the mBERT baseline,
and visualises entity-type confusion.

**Prerequisites**: Complete Phase 4 — `outputs/checkpoints/best_model/` must exist.

## 1. Setup

In [ ]:
import sys
import os
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from datasets import load_from_disk
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
)
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix

from dataset import id2label, LABEL_LIST
from utils import tokenizer

BEST_MODEL_PATH = REPO_ROOT / 'outputs' / 'checkpoints' / 'best_model'
RESULTS_DIR     = REPO_ROOT / 'outputs' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(
        f'Best model not found at {BEST_MODEL_PATH}.\n'
        'Complete Phase 4 training first.'
    )
print(f'Model path : {BEST_MODEL_PATH}')
print(f'Results dir: {RESULTS_DIR}')

## 2. Load Model and Prediction Helper

We load the best checkpoint saved at the end of Phase 4 and define a reusable
`predict_on_split()` helper that returns aligned true/predicted label sequences.

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(str(BEST_MODEL_PATH))
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    data_collator=data_collator,
)


def predict_on_split(lang, split='test'):
    """Run inference and return (true_labels, true_preds) as BIO string lists."""
    ds = load_from_disk(str(REPO_ROOT / 'data' / 'processed' / lang))
    preds_out = trainer.predict(ds[split])
    logits, labels = preds_out.predictions, preds_out.label_ids
    predictions = np.argmax(logits, axis=2)
    true_labels = [
        [id2label[l] for l in seq if l != -100]
        for seq in labels
    ]
    true_preds = [
        [id2label[p] for (p, l) in zip(ps, ls) if l != -100]
        for ps, ls in zip(predictions, labels)
    ]
    return true_labels, true_preds


print('Model loaded. Ready for evaluation.')

## 3. English Test Evaluation

seqeval evaluates at the entity span level: a prediction is correct only when
both the span boundary and entity type exactly match the gold annotation.

In [ ]:
print('Running inference on EN test set ...')
en_true, en_pred = predict_on_split('en', 'test')

print('\n=== English Test Results ===')
print(classification_report(en_true, en_pred))

en_report = classification_report(en_true, en_pred, output_dict=True)

# Build a readable per-entity table
en_rows = []
for entity in ['DIS', 'SYM', 'PRO']:
    row = en_report.get(entity, {})
    en_rows.append({
        'Entity': entity,
        'Precision': round(row.get('precision', 0), 4),
        'Recall':    round(row.get('recall',    0), 4),
        'F1':        round(row.get('f1-score',  0), 4),
        'Support':   int(row.get('support',     0)),
    })
en_df = pd.DataFrame(en_rows)
en_overall_f1 = f1_score(en_true, en_pred)
print(f'\nOverall entity F1 (EN): {en_overall_f1:.4f}')
en_df

## 4. Spanish Test Evaluation

The model was trained on the joint EN+ES dataset, so Spanish evaluation measures
zero-shot cross-lingual transfer quality.

In [ ]:
print('Running inference on ES test set ...')
es_true, es_pred = predict_on_split('es', 'test')

print('\n=== Spanish Test Results ===')
print(classification_report(es_true, es_pred))

es_report = classification_report(es_true, es_pred, output_dict=True)

es_rows = []
for entity in ['DIS', 'SYM', 'PRO']:
    row = es_report.get(entity, {})
    es_rows.append({
        'Entity': entity,
        'Precision': round(row.get('precision', 0), 4),
        'Recall':    round(row.get('recall',    0), 4),
        'F1':        round(row.get('f1-score',  0), 4),
        'Support':   int(row.get('support',     0)),
    })
es_df = pd.DataFrame(es_rows)
es_overall_f1 = f1_score(es_true, es_pred)
print(f'\nOverall entity F1 (ES): {es_overall_f1:.4f}')
es_df

## 5. Model Comparison Table

Side-by-side comparison of XLM-R vs the mBERT baseline.
mBERT baseline values are filled in after running `python src/train.py --model-name bert-base-multilingual-cased`.

In [ ]:
def extract_f1(report_dict, entity):
    return round(report_dict.get(entity, {}).get('f1-score', float('nan')), 4)

comparison_data = [
    {
        'Model':      'mBERT',
        'Language':   'EN',
        'DIS F1':     '—',
        'SYM F1':     '—',
        'PRO F1':     '—',
        'Overall F1': '— (run mBERT baseline)',
    },
    {
        'Model':      'mBERT',
        'Language':   'ES',
        'DIS F1':     '—',
        'SYM F1':     '—',
        'PRO F1':     '—',
        'Overall F1': '— (run mBERT baseline)',
    },
    {
        'Model':      'XLM-R (joint)',
        'Language':   'EN',
        'DIS F1':     extract_f1(en_report, 'DIS'),
        'SYM F1':     extract_f1(en_report, 'SYM'),
        'PRO F1':     extract_f1(en_report, 'PRO'),
        'Overall F1': round(en_overall_f1, 4),
    },
    {
        'Model':      'XLM-R (joint)',
        'Language':   'ES',
        'DIS F1':     extract_f1(es_report, 'DIS'),
        'SYM F1':     extract_f1(es_report, 'SYM'),
        'PRO F1':     extract_f1(es_report, 'PRO'),
        'Overall F1': round(es_overall_f1, 4),
    },
]

comp_df = pd.DataFrame(comparison_data)
print('Model comparison table (mBERT rows require separate training run):')
comp_df

## 5b. Ensemble Evaluation (XLM-R + mBERT Logit Averaging)

Once both XLM-R and mBERT checkpoints are trained, their raw output logits can be
averaged before argmax decoding. Logit averaging is mathematically preferable to
averaging softmax probabilities because it is equivalent to a geometric mean of the
output distributions.

**Expected gain**: +1–2% F1 over the single best model.

This cell runs automatically if the mBERT checkpoint exists at
`outputs/checkpoints/mbert_best/`; otherwise it prints a skip message.

In [ ]:
from evaluate import ensemble_predict

XLMR_PATH  = REPO_ROOT / 'outputs' / 'checkpoints' / 'best_model'
MBERT_PATH = REPO_ROOT / 'outputs' / 'checkpoints' / 'mbert_best'

if not MBERT_PATH.exists():
    print(f'mBERT checkpoint not found at {MBERT_PATH}.')
    print('Train mBERT first:  python src/train.py --model-name bert-base-multilingual-cased')
    print('Then save the best model to outputs/checkpoints/mbert_best/')
else:
    print('Running ensemble inference (XLM-R + mBERT logit averaging) ...')
    for lang, true_seqs in [('en', en_true), ('es', es_true)]:
        ds = load_from_disk(str(REPO_ROOT / 'data' / 'processed' / lang))
        avg_logits, label_ids = ensemble_predict(
            [str(XLMR_PATH), str(MBERT_PATH)],
            ds['test'],
            data_collator,
        )
        ens_preds = np.argmax(avg_logits, axis=2)
        ens_true  = [[id2label[l] for l in seq if l != -100] for seq in label_ids]
        ens_pred  = [
            [id2label[p] for (p, l) in zip(ps, ls) if l != -100]
            for ps, ls in zip(ens_preds, label_ids)
        ]

        ens_f1 = f1_score(ens_true, ens_pred)
        print(f'\n=== ENSEMBLE {lang.upper()} Test ===')
        print(classification_report(ens_true, ens_pred))
        print(f'Overall F1 : {ens_f1:.4f}')

        ens_report = classification_report(ens_true, ens_pred, output_dict=True)
        with open(RESULTS_DIR / f'{lang}_test_ensemble_results.json', 'w') as f:
            json.dump(ens_report, f, indent=2)

## 6. Entity-Type Confusion Matrix

Flatten all token predictions (ignoring `O`) to see how often the model confuses
one entity type for another — e.g., predicting `DIS` when the gold is `SYM`.

In [ ]:
def flatten_entity_tokens(true_seqs, pred_seqs):
    """Return flat lists of (true, pred) labels for non-O tokens only."""
    flat_true, flat_pred = [], []
    for ts, ps in zip(true_seqs, pred_seqs):
        for t, p in zip(ts, ps):
            if t != 'O' or p != 'O':  # include any token where true or pred is an entity
                # Normalise B-/I- prefix for confusion matrix (focus on type, not boundary)
                t_type = t[2:] if t != 'O' else 'O'
                p_type = p[2:] if p != 'O' else 'O'
                flat_true.append(t_type)
                flat_pred.append(p_type)
    return flat_true, flat_pred


en_ft, en_fp = flatten_entity_tokens(en_true, en_pred)
entity_labels = ['DIS', 'SYM', 'PRO', 'O']
cm = confusion_matrix(en_ft, en_fp, labels=entity_labels)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=entity_labels,
    yticklabels=entity_labels,
    ax=ax,
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Entity-Type Confusion Matrix (EN test, token level)')
plt.tight_layout()
cm_path = RESULTS_DIR / 'confusion_matrix_en.png'
plt.savefig(str(cm_path), dpi=150, bbox_inches='tight')
print(f'Saved: {cm_path}')
plt.show()

## 7. Save Results to Disk

In [ ]:
import json

with open(RESULTS_DIR / 'en_test_results.json', 'w') as f:
    json.dump(en_report, f, indent=2)
with open(RESULTS_DIR / 'es_test_results.json', 'w') as f:
    json.dump(es_report, f, indent=2)

print(f'EN results saved → {RESULTS_DIR / "en_test_results.json"}')
print(f'ES results saved → {RESULTS_DIR / "es_test_results.json"}')

## Summary

**What was done in this phase:**
- Evaluated the best XLM-R checkpoint on EN and ES test sets using seqeval entity-level F1
- Built a per-entity (DIS / SYM / PRO) F1 table for both languages
- Produced a model comparison table (XLM-R vs mBERT placeholder)
- Visualised entity-type confusion at the token level
- Saved JSON results to `outputs/results/`

**Next step**: Phase 5 (continued) — Error Analysis (`notebooks/05_error_analysis.ipynb`)